# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR² Croissant dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and includes multiple record sets, fields, and columns as defined by their `@id`.


In [ ]:
# Ensure mlcroissant is installed
!pip install -U mlcroissant pandas

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(url)
# Access metadata (as a Python object)
metadata = dataset.metadata
print(f"{getattr(metadata, 'name', None)}: {getattr(metadata, 'description', None)}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

In [ ]:
# Explore the record sets, fields, and columns via their @id
all_record_sets = [rs for rs in getattr(metadata, 'recordSet', [])]

print(f"Total record sets: {len(all_record_sets)}\n")
for i, record_set in enumerate(all_record_sets):
    rset_id = getattr(record_set, '@id', None)
    rset_name = getattr(record_set, 'name', rset_id)
    rset_desc = getattr(record_set, 'description', '')
    print(f"[{i}] Record set: {rset_name} (@id: {rset_id})")
    if rset_desc:
        print(f"    Description: {rset_desc}")
    # List fields for this record set
    fields = getattr(record_set, 'field', [])
    if not isinstance(fields, list):
        fields = [fields]
    print(f"    Fields:")
    for field in fields:
        fid = getattr(field, '@id', None)
        fname = getattr(field, 'name', fid)
        fdesc = getattr(field, 'description', '')
        print(f"     - {fname} (@id: {fid})    {f'| {fdesc}' if fdesc else ''}")
    print()

## 3. Data Extraction
Load data from all available record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# List the record set @id for easy access
record_set_ids = [getattr(rs, '@id', None) for rs in getattr(metadata, 'recordSet', [])]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records for record set '{record_set_id}'. Columns: {df.columns.tolist()}")
        else:
            print(f"No records found for record set '{record_set_id}'.")
    except Exception as ex:
        print(f"Could not load records for record set '{record_set_id}': {ex}")

# Example: Show columns and sample of the first non-empty record set
if dataframes:
    first_id = list(dataframes.keys())[0]
    print(f"\nColumns in {first_id}: \n", dataframes[first_id].columns.tolist())
    dataframes[first_id].head()

## 4. Exploratory Data Analysis (EDA)
You can now apply various data processing steps, such as filtering records by criteria, normalizing numeric fields, and grouping data for summary. We'll show an example by selecting the first numeric field in the first record set.

In [ ]:
# Automatically pick the first record set and first numeric-looking field
import numpy as np

if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    # Try to find a numeric field (column with dtype int/float) in the dataframe
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_cols:
        # Try to convert columns to float where possible
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except:
                continue
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if numeric_cols:
        numeric_field = numeric_cols[0]
        threshold = df[numeric_field].quantile(0.8)  # 80th percentile as example threshold
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to select a group field (e.g., a non-numeric column)
        group_fields = [c for c in df.columns if df[c].dtype == object]
        if group_fields:
            group_field = group_fields[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"\nGrouped mean {numeric_field} by '{group_field}':")
            print(grouped_df.head())
    else:
        print('No numeric fields detected in first record set. Please check the column types.')
else:
    print('No dataframes loaded. Please check dataset availability and Croissant record set definitions.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We'll visualize the distribution of the first numeric field (if available) for the first loaded record set.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'numeric_field' in locals():
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field} in record set '{record_set_id}'")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
This notebook demonstrated how to load, inspect, and analyze the FAIR² Croissant dataset about protein mass spectrometry experiments with `mlcroissant`.

Key steps included:
- Loading the metadata and record set structure by `@id`.
- Loading all available record sets into pandas DataFrames.
- Exploring fields and using IDs for reliable referencing.
- Filtering, normalizing, and visualizing numeric fields for exploratory data analysis.

For further analysis, see the Croissant schema documentation and [mlcroissant documentation](https://mlcommons.github.io/croissant/) for advanced usage.
